## Example 1: Basic HTTP Response Mocking

In [ ]:
# Example 1: Basic HTTP Response Mocking
import responses
import requests
import json

print("=== Basic HTTP Response Mocking ===")

# Basic responses configuration
@responses.activate
def test_basic_get_request():
    """Test basic GET request mocking"""
    # Add mock response
    responses.add(
        responses.GET,
        'https://api.example.com/users/1',
        json={'id': 1, 'name': 'John Doe', 'email': 'john@example.com'},
        status=200
    )
    
    # Make request
    resp = requests.get('https://api.example.com/users/1')
    return resp.json(), resp.status_code

# Test different response types
mock_responses = [
    {
        'method': responses.GET,
        'url': 'https://api.example.com/health',
        'json': {'status': 'healthy', 'timestamp': '2024-01-01T12:00:00Z'},
        'status': 200,
        'description': 'Health check endpoint'
    },
    {
        'method': responses.POST,
        'url': 'https://api.example.com/users',
        'json': {'id': 2, 'name': 'Jane Smith', 'created': True},
        'status': 201,
        'description': 'User creation endpoint'
    },
    {
        'method': responses.GET,
        'url': 'https://api.example.com/errors/500',
        'json': {'error': 'Internal server error', 'code': 500},
        'status': 500,
        'description': 'Server error simulation'
    }
]

print(f"Configured mock responses: {len(mock_responses)}")

for mock in mock_responses:
    print(f"  {mock['description']}:")
    print(f"    Method: {mock['method']}")
    print(f"    URL: {mock['url']}")
    print(f"    Status: {mock['status']}")

# Test basic mocking
result, status = test_basic_get_request()
print(f"\nBasic GET test result:")
print(f"  Response: {result}")
print(f"  Status: {status}")
assert result == {'id': 1, 'name': 'John Doe', 'email': 'john@example.com'}, result
assert status == 200, status

# Test multiple mocked endpoints
@responses.activate
def test_multiple_endpoints():
    """Test multiple mocked endpoints"""
    results = []
    
    for mock in mock_responses:
        responses.add(
            mock['method'],
            mock['url'],
            json=mock['json'],
            status=mock['status']
        )
    
    # Test each endpoint: each one must answer with the status and body it was registered with
    for mock in mock_responses:
        if mock['method'] == responses.GET:
            resp = requests.get(mock['url'])
        else:
            resp = requests.post(mock['url'], json={'name': 'Jane Smith'})

        results.append({
            'url': mock['url'],
            'status': resp.status_code,
            'success': resp.status_code == mock['status'] and resp.json() == mock['json']
        })

    return results

# Test multiple endpoints
multi_results = test_multiple_endpoints()
print(f"\nMultiple endpoints test:")
for result in multi_results:
    status_text = "✓" if result['success'] else "✗"
    print(f"  {status_text} {result['url']}: {result.get('status', 'ERROR')}")
assert all(r['success'] for r in multi_results), multi_results
assert [r['status'] for r in multi_results] == [200, 201, 500], multi_results

print("Basic HTTP response mocking completed!")

## Example 2: Advanced Response Patterns and Dynamic Responses

In [ ]:
# Example 2: Advanced Response Patterns and Dynamic Responses
import responses
import requests
from responses import Response, CallbackResponse

print("=== Advanced Response Patterns ===")

# Custom response headers and content types
@responses.activate
def test_custom_headers():
    """Test responses with custom headers"""
    responses.add(
        responses.GET,
        'https://api.example.com/data',
        json={'data': 'test'},
        headers={
            'Content-Type': 'application/json',
            'X-Rate-Limit': '100',
            'X-API-Version': '1.0',
            'Cache-Control': 'no-cache'
        },
        status=200
    )
    
    resp = requests.get('https://api.example.com/data')
    return resp.json(), dict(resp.headers)

# Test different content types
@responses.activate  
def test_content_types():
    """Test different response content types"""
    
    # JSON response
    responses.add(
        responses.GET,
        'https://api.example.com/json',
        json={'type': 'json', 'valid': True}
    )
    
    # Plain text response
    responses.add(
        responses.GET,
        'https://api.example.com/text',
        body='This is plain text content',
        content_type='text/plain'
    )
    
    # XML response
    responses.add(
        responses.GET,
        'https://api.example.com/xml',
        body='<?xml version="1.0"?><root><item>value</item></root>',
        content_type='application/xml'
    )
    
    results = []
    
    # Test each content type
    test_urls = [
        ('https://api.example.com/json', 'json'),
        ('https://api.example.com/text', 'text'),
        ('https://api.example.com/xml', 'xml')
    ]
    
    for url, content_type in test_urls:
        resp = requests.get(url)
        results.append({
            'type': content_type,
            'status': resp.status_code,
            'content_type': resp.headers.get('Content-Type', 'unknown'),
            'content_length': len(resp.text)
        })
    
    return results

# Dynamic callback responses
def create_dynamic_callback(base_data):
    """Create a dynamic callback function"""
    def callback(request):
        # Extract query parameters
        query_params = {}
        if '?' in request.url:
            query_string = request.url.split('?')[1]
            for param in query_string.split('&'):
                if '=' in param:
                    key, value = param.split('=', 1)
                    query_params[key] = value
        
        # Create dynamic response based on request
        response_data = base_data.copy()
        response_data.update({
            'request_method': request.method,
            'query_params': query_params,
            'timestamp': '2024-01-01T12:00:00Z'
        })
        
        return (200, {}, json.dumps(response_data))
    
    return callback

print("Testing custom headers and content types:")

# Test custom headers
json_data, headers = test_custom_headers()
print(f"  Custom headers response:")
print(f"    Data: {json_data}")
print(f"    Key headers: X-Rate-Limit={headers.get('X-Rate-Limit')}, X-API-Version={headers.get('X-API-Version')}")

# Test content types  
content_results = test_content_types()
print(f"  Content types test:")
for result in content_results:
    print(f"    {result['type']}: {result['content_type']}, {result['content_length']} bytes")

# Test dynamic callback responses
@responses.activate
def test_dynamic_responses():
    """Test dynamic callback responses"""
    
    # Add callback response
    responses.add_callback(
        responses.GET,
        'https://api.example.com/dynamic',
        callback=create_dynamic_callback({'service': 'dynamic', 'version': '1.0'}),
        content_type='application/json'
    )
    
    # Test different requests to same endpoint
    test_requests = [
        'https://api.example.com/dynamic',
        'https://api.example.com/dynamic?user=123',
        'https://api.example.com/dynamic?user=456&action=login'
    ]
    
    results = []
    for url in test_requests:
        resp = requests.get(url)
        data = resp.json()
        results.append({
            'url': url,
            'query_params': data.get('query_params', {}),
            'method': data.get('request_method')
        })
    
    return results

print(f"\nTesting dynamic callback responses:")
dynamic_results = test_dynamic_responses()
for i, result in enumerate(dynamic_results, 1):
    print(f"  Request {i}:")
    print(f"    URL: {result['url']}")
    print(f"    Method: {result['method']}")
    print(f"    Query params: {result['query_params']}")

print("Advanced response patterns completed!")

## Example 3: Request Validation and Call Introspection

In [ ]:
# Example 3: Request Validation and Call Introspection
import responses
import requests
import json

print("=== Request Validation and Call Introspection ===")

# Request validation with matchers
@responses.activate
def test_request_validation():
    """Test request validation using matchers"""
    
    # Add response with JSON matcher
    responses.add(
        responses.POST,
        'https://api.example.com/users',
        json={'success': True, 'id': 123},
        match=[
            responses.matchers.json_params_matcher({
                'name': 'John Doe',
                'email': 'john@example.com'
            })
        ]
    )
    
    # Valid request
    valid_payload = {'name': 'John Doe', 'email': 'john@example.com'}
    
    resp = requests.post('https://api.example.com/users', json=valid_payload)
    valid_result = {
        'status': resp.status_code,
        'response': resp.json(),
        'success': True
    }

    # Invalid request (will fail validation)
    invalid_payload = {'name': 'Jane Smith', 'email': 'jane@example.com'}
    
    # responses refuses a request that no registered mock matches (the json_params_matcher rejects this
    # body) by raising requests' ConnectionError; anything else, or no error at all, is a defect
    try:
        resp = requests.post('https://api.example.com/users', json=invalid_payload)
    except requests.exceptions.ConnectionError as e:
        assert "Connection refused by Responses" in str(e), e
        invalid_result = {'success': False, 'error': str(e)}
    else:
        raise AssertionError(f"json_params_matcher accepted a non-matching payload (HTTP {resp.status_code})")
    
    return valid_result, invalid_result

# Call introspection and verification
@responses.activate  
def test_call_introspection():
    """Test call introspection and verification"""
    
    # Set up multiple mock endpoints
    mock_endpoints = [
        ('GET', 'https://api.example.com/users'),
        ('POST', 'https://api.example.com/users'),
        ('GET', 'https://api.example.com/users/123'),
        ('DELETE', 'https://api.example.com/users/123')
    ]
    
    for method, url in mock_endpoints:
        responses.add(
            getattr(responses, method),
            url,
            json={'method': method, 'url': url, 'success': True}
        )
    
    # Make test requests
    requests.get('https://api.example.com/users')
    requests.post('https://api.example.com/users', json={'name': 'Test User'})
    requests.get('https://api.example.com/users/123')
    
    # Introspect calls
    call_data = []
    for call in responses.calls:
        call_data.append({
            'method': call.request.method,
            'url': call.request.url,
            'has_body': bool(call.request.body),
            'response_status': call.response.status_code
        })
    
    return call_data, len(responses.calls)

# URL pattern matching
@responses.activate
def test_url_patterns():
    """Test URL pattern matching"""
    import re
    
    # Exact URL match
    responses.add(
        responses.GET,
        'https://api.example.com/users/123',
        json={'id': 123, 'type': 'exact'}
    )
    
    # Regex pattern match
    responses.add(
        responses.GET,
        re.compile(r'https://api\.example\.com/users/\d+/posts'),
        json={'type': 'posts', 'pattern': 'regex'}
    )
    
    # Test different URL patterns
    test_urls = [
        'https://api.example.com/users/123',
        'https://api.example.com/users/456/posts',
        'https://api.example.com/users/789/posts'
    ]
    
    results = []
    for url in test_urls:
        resp = requests.get(url)
        results.append({
            'url': url,
            'status': resp.status_code,
            'response_type': resp.json().get('type'),
            'matched': resp.status_code == 200
        })

    return results

print("Testing request validation:")

# Test request validation
valid_result, invalid_result = test_request_validation()
print(f"  Valid request:")
print(f"    Success: {valid_result['success']}")
if valid_result['success']:
    print(f"    Response: {valid_result['response']}")
    print(f"    Status: {valid_result['status']}")

print(f"  Invalid request:")
print(f"    Success: {invalid_result['success']}")
if not invalid_result['success']:
    print(f"    Refused as expected - {invalid_result['error'].splitlines()[0]}")
assert valid_result['status'] == 200, valid_result
assert valid_result['response'] == {'success': True, 'id': 123}, valid_result

print(f"\nTesting call introspection:")

# Test call introspection
call_data, total_calls = test_call_introspection()
print(f"  Total calls made: {total_calls}")
print(f"  Call details:")
for i, call in enumerate(call_data, 1):
    body_info = "with body" if call['has_body'] else "no body"
    print(f"    {i}. {call['method']} {call['url']} ({body_info}) -> {call['response_status']}")
assert total_calls == 3, total_calls
assert [(call['method'], call['url'], call['has_body']) for call in call_data] == [
    ('GET', 'https://api.example.com/users', False),
    ('POST', 'https://api.example.com/users', True),
    ('GET', 'https://api.example.com/users/123', False),
], call_data

print(f"\nTesting URL pattern matching:")

# Test URL patterns
pattern_results = test_url_patterns()
print(f"  Pattern matching results:")
for result in pattern_results:
    if result['matched']:
        print(f"    ✓ {result['url']} -> {result['response_type']} (status: {result['status']})")
    else:
        print(f"    ✗ {result['url']} -> No match")
assert all(r['matched'] for r in pattern_results), pattern_results
assert [r['response_type'] for r in pattern_results] == ['exact', 'posts', 'posts'], pattern_results

# Test assertion helpers
print(f"\nTesting assertion helpers:")

@responses.activate
def test_assertions():
    """Test responses assertion helpers"""
    responses.add(responses.GET, 'https://api.example.com/test', json={'test': True})
    
    # Make request
    requests.get('https://api.example.com/test')
    
    # Raises AssertionError unless the URL was called exactly once
    responses.assert_call_count('https://api.example.com/test', 1)
    return "Call count assertion passed"

assertion_result = test_assertions()
print(f"  {assertion_result}")

print("Request validation and introspection completed!")

## Example 4: Response Management and Advanced Mock Configuration

In [ ]:
# Example 4: Response Management and Advanced Mock Configuration
import responses
import requests
import time
from unittest.mock import patch

print("=== Response Management and Advanced Configuration ===")

# Response registry management
@responses.activate
def test_response_management():
    """Test response registry management"""
    
    # Add initial responses
    responses.add(responses.GET, 'https://api.example.com/v1/users', json={'version': '1.0'})
    responses.add(responses.GET, 'https://api.example.com/v2/users', json={'version': '2.0'})
    
    print(f"  Initial registered responses: {len(responses.registered())}")
    
    # Test initial responses
    resp_v1 = requests.get('https://api.example.com/v1/users')
    resp_v2 = requests.get('https://api.example.com/v2/users')
    
    initial_results = {
        'v1': resp_v1.json(),
        'v2': resp_v2.json(),
        'registered_count': len(responses.registered())
    }
    
    # Replace a response
    responses.replace(
        responses.GET,
        'https://api.example.com/v1/users',
        json={'version': '1.1', 'updated': True}
    )
    
    # Test replaced response
    resp_v1_updated = requests.get('https://api.example.com/v1/users')
    
    # Remove a response
    responses.remove(responses.GET, 'https://api.example.com/v2/users')
    
    final_results = {
        'v1_updated': resp_v1_updated.json(),
        'registered_after_replace': len(responses.registered()),
        'registered_after_remove': len(responses.registered())
    }
    
    return initial_results, final_results

# Multiple response variants for same endpoint
@responses.activate
def test_multiple_variants():
    """Test multiple response variants for same endpoint"""
    
    # Add multiple responses for same URL (will be served in order)
    response_variants = [
        {'json': {'status': 'loading', 'progress': 25}, 'status': 202},
        {'json': {'status': 'processing', 'progress': 75}, 'status': 202},
        {'json': {'status': 'complete', 'data': {'id': 123}}, 'status': 200}
    ]
    
    for variant in response_variants:
        responses.add(
            responses.GET,
            'https://api.example.com/task/status',
            json=variant['json'],
            status=variant['status']
        )
    
    # Make multiple requests (each gets next response)
    results = []
    for i in range(3):
        resp = requests.get('https://api.example.com/task/status')
        results.append({
            'request': i + 1,
            'status': resp.status_code,
            'data': resp.json()
        })
    
    return results

# Response with delays and streaming
class MockResponse:
    """Mock response class for advanced testing"""
    
    @staticmethod
    def create_delayed_response():
        """Create response that simulates network delay"""
        def callback(request):
            # Simulate processing delay
            time.sleep(0.1)  # 100ms delay
            return (200, {}, json.dumps({
                'processed_at': time.time(),
                'request_url': request.url,
                'delay_simulated': True
            }))
        return callback
    
    @staticmethod
    def create_error_response():
        """Create response that simulates server errors"""
        def callback(request):
            # Simulate different error conditions
            if 'timeout' in request.url:
                raise requests.exceptions.Timeout("Simulated timeout")
            elif 'connection' in request.url:
                raise requests.exceptions.ConnectionError("Simulated connection error")
            else:
                return (500, {}, json.dumps({'error': 'Internal server error'}))
        return callback

@responses.activate
def test_advanced_mocking():
    """Test advanced mocking scenarios"""
    
    # Add delayed response
    responses.add_callback(
        responses.GET,
        'https://api.example.com/slow',
        callback=MockResponse.create_delayed_response()
    )
    
    # Add error responses
    responses.add_callback(
        responses.GET,
        'https://api.example.com/error/timeout',
        callback=MockResponse.create_error_response()
    )
    
    responses.add_callback(
        responses.GET,
        'https://api.example.com/error/connection',
        callback=MockResponse.create_error_response()
    )
    
    results = []
    
    # Test delayed response
    start_time = time.time()
    try:
        resp = requests.get('https://api.example.com/slow')
        elapsed = time.time() - start_time
        results.append({
            'type': 'delayed',
            'success': True,
            'elapsed': elapsed,
            'data': resp.json()
        })
    except Exception as e:
        results.append({
            'type': 'delayed',
            'success': False,
            'error': str(e)
        })
    
    # Test error responses
    error_urls = [
        ('https://api.example.com/error/timeout', 'timeout'),
        ('https://api.example.com/error/connection', 'connection')
    ]
    
    # the error callback raises these on purpose; each URL must surface exactly its own exception
    expected_errors = {
        'timeout': requests.exceptions.Timeout,
        'connection': requests.exceptions.ConnectionError,
    }
    
    for url, error_type in error_urls:
        try:
            resp = requests.get(url)
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as e:
            assert type(e) is expected_errors[error_type], (url, type(e))
            results.append({
                'type': error_type,
                'success': False,
                'error': type(e).__name__
            })
        else:
            raise AssertionError(f"{url}: the error callback returned HTTP {resp.status_code} instead of raising")
    
    return results

print("Testing response management:")

# Test response management
initial_results, final_results = test_response_management()
print(f"  Initial state:")
print(f"    V1 response: {initial_results['v1']}")
print(f"    V2 response: {initial_results['v2']}")
print(f"    Registered: {initial_results['registered_count']}")

print(f"  After management:")
print(f"    V1 updated: {final_results['v1_updated']}")
print(f"    Registered after operations: {final_results['registered_after_remove']}")

print(f"\nTesting multiple response variants:")

# Test multiple variants
variant_results = test_multiple_variants()
for result in variant_results:
    status_info = result['data']['status']
    progress = result['data'].get('progress', 'N/A')
    print(f"  Request {result['request']}: {status_info} (progress: {progress}) - Status {result['status']}")

print(f"\nTesting advanced mocking scenarios:")

# Test advanced mocking
advanced_results = test_advanced_mocking()
for result in advanced_results:
    if result['success']:
        if result['type'] == 'delayed':
            print(f"  {result['type']}: Success, elapsed {result['elapsed']:.3f}s")
        else:
            print(f"  {result['type']}: Success, status {result['status']}")
    else:
        print(f"  {result['type']}: {result['error']}")

print("Response management and advanced configuration completed!")

## Example 5: Testing Frameworks Integration and Production Patterns

In [ ]:
# Example 5: Testing Frameworks Integration and Production Patterns
import responses
import requests
import unittest
from contextlib import contextmanager

print("=== Testing Frameworks Integration ===")

# Context manager usage
@contextmanager
def mock_api():
    """Context manager for API mocking"""
    responses.start()
    try:
        # Add common API responses
        responses.add(
            responses.GET,
            'https://api.example.com/status',
            json={'status': 'ok', 'service': 'test-api'}
        )
        responses.add(
            responses.GET,
            'https://api.example.com/config',
            json={'version': '1.0', 'environment': 'test'}
        )
        yield
    finally:
        responses.stop()
        responses.reset()

# Test class structure for unittest integration
class APITestSuite:
    """Test suite demonstrating responses usage patterns"""
    
    @responses.activate
    def test_user_creation(self):
        """Test user creation API endpoint"""
        # Mock the API response
        responses.add(
            responses.POST,
            'https://api.example.com/users',
            json={'id': 123, 'name': 'Test User', 'created': True},
            status=201
        )
        
        # Test the functionality
        user_data = {'name': 'Test User', 'email': 'test@example.com'}
        resp = requests.post('https://api.example.com/users', json=user_data)
        
        return {
            'success': resp.status_code == 201,
            'response': resp.json(),
            'calls_made': len(responses.calls)
        }
    
    @responses.activate
    def test_error_handling(self):
        """Test API error handling"""
        # Mock error responses
        error_scenarios = [
            (400, {'error': 'Bad Request', 'message': 'Invalid data'}),
            (401, {'error': 'Unauthorized', 'message': 'Authentication required'}),
            (404, {'error': 'Not Found', 'message': 'Resource not found'}),
            (500, {'error': 'Internal Server Error', 'message': 'Server error'})
        ]
        
        results = []
        for status_code, error_data in error_scenarios:
            responses.add(
                responses.GET,
                f'https://api.example.com/error/{status_code}',
                json=error_data,
                status=status_code
            )
            
            resp = requests.get(f'https://api.example.com/error/{status_code}')
            results.append({
                'status_code': status_code,
                'error_handled': resp.status_code == status_code,
                'error_data': resp.json()
            })
        
        return results
    
    def test_context_manager_usage(self):
        """Test using context manager for mocking"""
        results = []
        
        with mock_api():
            # Test API endpoints within context
            status_resp = requests.get('https://api.example.com/status')
            config_resp = requests.get('https://api.example.com/config')
            
            results.extend([
                {
                    'endpoint': 'status',
                    'success': status_resp.status_code == 200,
                    'data': status_resp.json()
                },
                {
                    'endpoint': 'config',
                    'success': config_resp.status_code == 200,
                    'data': config_resp.json()
                }
            ])
        
        return results

# Production testing patterns
class ProductionTestPatterns:
    """Production-ready testing patterns with responses"""
    
    @staticmethod
    @responses.activate
    def test_api_integration_flow():
        """Test complete API integration flow"""
        # Set up mock responses for entire workflow
        workflow_responses = [
            # Authentication
            {
                'method': responses.POST,
                'url': 'https://api.example.com/auth/login',
                'response': {'token': 'abc123', 'expires_in': 3600},
                'status': 200
            },
            # Data retrieval
            {
                'method': responses.GET,
                'url': 'https://api.example.com/data/users',
                'response': {'users': [{'id': 1, 'name': 'Alice'}, {'id': 2, 'name': 'Bob'}]},
                'status': 200
            },
            # Data processing result
            {
                'method': responses.POST,
                'url': 'https://api.example.com/process',
                'response': {'job_id': 'job_123', 'status': 'completed'},
                'status': 200
            }
        ]
        
        # Register all responses
        for mock in workflow_responses:
            responses.add(
                mock['method'],
                mock['url'],
                json=mock['response'],
                status=mock['status']
            )
        
        # Execute workflow
        workflow_results = []
        
        # Step 1: Authentication
        auth_resp = requests.post('https://api.example.com/auth/login', 
                                 json={'username': 'test', 'password': 'test'})
        workflow_results.append({
            'step': 'authentication',
            'success': auth_resp.status_code == 200,
            'token': auth_resp.json().get('token')
        })
        
        # Step 2: Data retrieval
        headers = {'Authorization': f'Bearer {auth_resp.json().get("token")}'}
        data_resp = requests.get('https://api.example.com/data/users', headers=headers)
        workflow_results.append({
            'step': 'data_retrieval',
            'success': data_resp.status_code == 200,
            'user_count': len(data_resp.json().get('users', []))
        })
        
        # Step 3: Data processing
        process_resp = requests.post('https://api.example.com/process',
                                   json={'data': data_resp.json()},
                                   headers=headers)
        workflow_results.append({
            'step': 'processing',
            'success': process_resp.status_code == 200,
            'job_id': process_resp.json().get('job_id')
        })
        
        return workflow_results
    
    @staticmethod
    def test_performance_simulation():
        """Test performance characteristics with mocked responses"""
        import time
        
        # Performance test scenarios
        scenarios = [
            {'name': 'fast', 'delay': 0.05, 'count': 10},
            {'name': 'medium', 'delay': 0.1, 'count': 5},
            {'name': 'slow', 'delay': 0.2, 'count': 3}
        ]
        
        results = []
        
        for scenario in scenarios:
            with responses.RequestsMock() as rsps:
                def delayed_callback(request):
                    time.sleep(scenario['delay'])
                    return (200, {}, json.dumps({
                        'scenario': scenario['name'],
                        'processed_at': time.time()
                    }))
                
                rsps.add_callback(
                    responses.GET,
                    f'https://api.example.com/perf/{scenario["name"]}',
                    callback=delayed_callback
                )
                
                # Run performance test
                start_time = time.time()
                for i in range(scenario['count']):
                    resp = requests.get(f'https://api.example.com/perf/{scenario["name"]}')
                
                total_time = time.time() - start_time
                results.append({
                    'scenario': scenario['name'],
                    'requests': scenario['count'],
                    'total_time': round(total_time, 3),
                    'avg_time': round(total_time / scenario['count'], 3)
                })
        
        return results

# Run test examples
print("Testing frameworks integration:")

# Initialize test suite
test_suite = APITestSuite()

# Test user creation
user_test = test_suite.test_user_creation()
print(f"  User creation test:")
print(f"    Success: {user_test['success']}")
print(f"    Response: {user_test['response']}")
print(f"    Calls made: {user_test['calls_made']}")

# Test error handling  
error_tests = test_suite.test_error_handling()
print(f"  Error handling tests:")
for test in error_tests:
    print(f"    Status {test['status_code']}: {'✓' if test['error_handled'] else '✗'}")

# Test context manager usage
context_tests = test_suite.test_context_manager_usage()
print(f"  Context manager tests:")
for test in context_tests:
    print(f"    {test['endpoint']}: {'✓' if test['success'] else '✗'} - {test['data'].get('status', 'N/A')}")

print(f"\nProduction testing patterns:")

# Test integration flow
integration_results = ProductionTestPatterns.test_api_integration_flow()
print(f"  Integration workflow:")
for result in integration_results:
    status = '✓' if result['success'] else '✗'
    print(f"    {result['step']}: {status}")

# Test performance simulation
perf_results = ProductionTestPatterns.test_performance_simulation()
print(f"  Performance simulation:")
for result in perf_results:
    print(f"    {result['scenario']}: {result['requests']} requests in {result['total_time']}s (avg: {result['avg_time']}s)")

print("Testing frameworks integration completed!")